In [ ]:

from model import *
from train import train
from train import callbacks
import orbax

In [ ]:
train_dataloader

eval_dataloader

In [ ]:
trainer = 

In [ ]:
train.run(
  train_dataloader=train_dataloader,
  trainer=trainer,
  workdir="./",
  total_train_steps=1000,
  metric_aggregation_steps=50,
  eval_dataloader=eval_dataloader,
  eval_every_steps=2000,
  num_batches_per_eval=10,
  callbacks=[
    callbacks.TrainStateCheckpoint(
        base_dir="./",  # NOTE: this must be a full path
        options=orbax.checkpoint.CheckpointManagerOptions(
            save_interval_steps=500,
            max_to_keep=5,
        )
    ),
    callbacks.TqdmProgressBar(
        total_train_steps=1000,
        train_monitors=("train_loss",),
        eval_monitors=("eval_reconstruction_rel_l2",)),
  ],
)

In [ ]:
from data.beltramiDataGeneration import generate_da_data
import matplotlib.pyplot as plt

# Parameters
A_PARAM = 1.0
D_PARAM = 1.0
SPATIAL_RES = 16  # Reduced for quick demonstration
TIME_STEPS = 20
DT = 0.05

# Generate Data
truth, obs, coords, t_array = generate_da_data(
    spatial_res=SPATIAL_RES,
    time_steps=TIME_STEPS,
    dt=DT,
    a=A_PARAM,
    d=D_PARAM
)

# Shapes
print("\n--- Data Generation Complete ---")
print(f"Ground Truth Shape: {truth.shape} (Time, X, Y, Z, Vars)")
print(f"Variables channel index: 0:u, 1:v, 2:w, 3:p")

# --- Visualization (Optional Check) ---
# Plotting the U-velocity field at the middle Z-slice at t=0
z_idx = SPATIAL_RES // 2
u_field_t0 = truth[0, :, :, z_idx, 0] # Time 0, All X, All Y, Mid Z, Var U

plt.figure(figsize=(6, 5))
plt.contourf(coords['x'], coords['y'], u_field_t0, levels=50, cmap='viridis')
plt.colorbar(label='u velocity')
plt.title(f'Beltrami Flow: U-Velocity (z=0, t=0)\na={A_PARAM}, d={D_PARAM}')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
import optax
import os
from train import callbacks as cb

from data.beltramiDataGeneration import *
from model.unet3d import UNet3d
from train.models import DataAssimilationModel
from train.trainers import BeltramiTrainer
from train.train import run

if __name__=="__main__":

    # 1. Generate the Dataset
    # Using spatial_res=32 to fit in memory easily
    gt, obs, coords, times = generate_da_data(spatial_res=32, time_steps=100)

    # 1. Setup Architecture
    unet_config = UNet3d(
        out_channels=4,      # Required: [u, v, w, p]
        time_cond=True,      # Pass this explicitly to satisfy the error
        num_channels=(32, 64, 128, 256),
        use_spatial_attention=(False, False, True, True)
    )

    # 2. Update the Model Wrapper accordingly
    da_model = DataAssimilationModel(
        model=unet_config, 
        input_shape=(4, 32, 32, 32, 4) 
    )

    # 4. Define Optimizer & Trainer
    tx = optax.adamw(learning_rate=1e-4, weight_decay=1e-5)
    trainer = BeltramiTrainer(
        model=da_model, 
        optimizer=tx, 
        rng=jax.random.PRNGKey(0)
    )

    # 5. Define Callbacks
    base_workdir = "./beltrami_experiment"
    workdir = os.path.abspath(base_workdir)  # Converts to absolute path
    callbacks = [
        cb.TqdmProgressBar(
            total_train_steps=5000, 
            train_monitors=["train_loss"]
        ),
        cb.TrainStateCheckpoint(base_dir=workdir)
    ]

    # 6. Start Training
    run(
        train_dataloader=beltrami_dataloader(obs, times, batch_size=4),
        trainer=trainer,
        workdir=workdir,
        total_train_steps=5000,
        metric_aggregation_steps=50,
        callbacks=callbacks
    )

Generating Ground Truth for grid 32x32x32 over 100 steps...
Adding observation noise...


  2%|▏         | 100/5000 [06:23<5:24:32,  3.97s/step, train_loss=0.04134954199670184]WARNING:absl:[process=0][thread=async_save] Skipping merge of OCDBT checkpoints: No per-process OCDBT checkpoint subdirs found in /Users/user/Code/[InProg] UQ/vit_beltrami/beltrami_experiment/checkpoints/100.orbax-checkpoint-tmp/default.orbax-checkpoint-tmp, 


# Extra

In [ ]:
import jax
import jax.numpy as jnp
from flax.core import FrozenDict

from model.unet3d import UNet3d


key = jax.random.PRNGKey(0)

batch = 2
nx, ny, nz = 16, 16, 16   # 3D grid
channels = 3               # e.g., velocity components

x = jax.random.normal(key, (batch, nx, ny, nz, channels))
sigma = jnp.ones((batch,)) * 0.5
time = jnp.linspace(0.0, 1.0, batch)

model = UNet3d(
    out_channels=channels,
    time_cond=True,
    num_channels = (16, 32),
    downsample_ratio = (2, 2),
    num_blocks=2,
    noise_embed_dim=32,
    input_proj_channels=16,
    output_proj_channels=16,
    padding="SAME",
    use_spatial_attention=(False, True),
)

variables = model.init(
    key,
    x,
    sigma,
    time,
    is_training=True,
)

params = variables["params"]
print("Parameter shapes:")
jax.tree_util.tree_map(lambda a: a.shape, params)

y = model.apply(
    variables,
    x,
    sigma,
    time,
    is_training=False,
)

print("Output shape:", y.shape)